# 08. Group-aware sampling and semantic phase origins

![Group-aware sampling](../images/08_group_aware_sampling.svg)

![Windows and transforms](../images/08_windows_and_transforms.svg)

This notebook builds a synthetic version of the phase catalog used by the iso-catalog study. Read the [lecture](../lectures/08_group_aware_sampling.md) and return to the [tutorial index](../README.md).

**Learning goals:** split groups before sampling clips, select nested phase origins from a common eligible corpus, construct a matched nearby-jitter control, and audit semantic separation rather than counting only start indices.

In [ ]:
import hashlib
import numpy as np

SEED = 23
rng = np.random.default_rng(SEED)
assert rng.random() >= 0.0


## Groups first

Windows from one source sequence must remain in one split. The synthetic identifiers below stand in for source groups, not people.

In [ ]:
groups = np.repeat(np.arange(12), 5)
test_groups = {1, 4, 8}
train = groups[~np.isin(groups, list(test_groups))]
test = groups[np.isin(groups, list(test_groups))]
assert set(train).isdisjoint(set(test))


## Nested semantic origins

A base phase is chosen by a stable hash. One, two, and four origins are nested. Nearby jitter uses four offsets around the same base, so the comparison changes separation rather than the base-phase distribution.

In [ ]:
def stable_uniform(*parts):
    payload = '|'.join(map(str, parts)).encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], 'big') / 2**64

def semantic_origins(sequence_id, block, k):
    base = stable_uniform('phase-v1', sequence_id, block)
    offsets = {1: (0.0,), 2: (0.0, 0.5), 4: (0.0, 0.25, 0.5, 0.75)}[k]
    return tuple((base + offset) % 1.0 for offset in offsets)

def nearby_jitter(sequence_id, block, radius=0.04):
    base = stable_uniform('phase-v1', sequence_id, block)
    return tuple((base + offset) % 1.0 for offset in (-radius, -radius / 3, radius / 3, radius))

one = semantic_origins('s07', 0, 1)
two = semantic_origins('s07', 0, 2)
four = semantic_origins('s07', 0, 4)
jitter = nearby_jitter('s07', 0)
assert one[0] == two[0] == four[0]
assert len(four) == len(jitter) == 4


## Audit the intended separation

Here circular phase distance is a synthetic proxy for pose-trajectory separation. A real audit uses frozen silhouette features and blinded manual validation.

In [ ]:
def circular_distance(a, b):
    delta = abs(a - b) % 1.0
    return min(delta, 1.0 - delta)

def trajectory_separation(origins):
    pairs = [circular_distance(a, b) for i, a in enumerate(origins) for b in origins[i + 1:]]
    return float(np.mean(pairs))

semantic_sep = trajectory_separation(four)
jitter_sep = trajectory_separation(jitter)
assert semantic_sep > jitter_sep
print('semantic separation', semantic_sep, 'jitter separation', jitter_sep)


## Iso-catalog allocation

All three path cells have the same nominal sequence-origin cardinality. This controls counting, not information content.

In [ ]:
allocations = {
    'breadth': (250_000, 1, 'base_phase'),
    'balanced': (125_000, 2, 'phase_separated'),
    'phase_depth': (62_500, 4, 'phase_separated'),
    'nearby_jitter': (62_500, 4, 'nearby_jitter'),
}
for name, (unique_sequences, origins_per_sequence, policy) in allocations.items():
    nominal_catalog_size = unique_sequences * origins_per_sequence
    assert nominal_catalog_size == 250_000
    print(name, policy, nominal_catalog_size)

phase_catalog = {'version': 'phase-v1', 'policy': 'phase_separated', 'threshold': 0.12}
phase_catalog_digest = hashlib.sha256(repr(sorted(phase_catalog.items())).encode()).hexdigest()
assert len(phase_catalog_digest) == 64


## Preserve the audit with the sample

A treatment label is insufficient. Store the catalog digest and the measurements that show what changed.

In [ ]:
required_audit_fields = {
    'phase_catalog_digest', 'origin_policy', 'nominal_catalog_size',
    'origin_coverage', 'window_overlap', 'trajectory_separation',
    'effective_cluster_count',
}
audit_record = {
    'phase_catalog_digest': phase_catalog_digest, 'origin_policy': 'phase_separated',
    'nominal_catalog_size': 250_000, 'origin_coverage': 4,
    'window_overlap': 0.5, 'trajectory_separation': semantic_sep,
    'effective_cluster_count': 61_000,
}
assert set(audit_record) == required_audit_fields
assert audit_record['trajectory_separation'] > jitter_sep


**Takeaway:** group-aware sampling prevents leakage, but a phase intervention needs more. It needs common eligibility, nested origin sets, a matched jitter control, and an audit that demonstrates semantic separation.

Previous: [07. Gradient updates](07_gradient_updates_and_schedules.ipynb) · Next: [09. Eigenspectra](09_eigenspectra_and_effective_rank.ipynb)